In [3]:
import pandas as pd
import tensorflow as tf
from datasets import Dataset
import numpy as np

# Load data
df_train = pd.read_csv("/kaggle/input/contradictory-my-dear-watson/train.csv")
df_test = pd.read_csv("/kaggle/input/contradictory-my-dear-watson/test.csv")

In [6]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    AutoConfig
)


# Load tokenizer
model_name = 'joeddav/xlm-roberta-large-xnli'

config = AutoConfig.from_pretrained(model_name)
config.hidden_dropout_prob = 0.3
config.attention_probs_dropout_prob = 0.3
config.num_labels = 3

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# import os
# os.environ["WANDB_DISABLED"] = "true"

In [7]:
# Preprocess dataset
def preprocess_function(examples):
    return tokenizer(
        examples['premise'], 
        examples['hypothesis'], 
        truncation=True, 
        padding="max_length", 
        max_length=150
    )

# Huggingface wants a Dataset object
train_dataset = Dataset.from_pandas(df_train)
train_dataset = train_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/12120 [00:00<?, ? examples/s]

In [8]:
# Remove columns Trainer doesn't need
train_dataset = train_dataset.remove_columns(["premise", "hypothesis", "id"])

# Huggingface expects labels column
train_dataset = train_dataset.rename_column("label", "labels")

In [9]:
from sklearn.model_selection import train_test_split

train_val = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_val['train']
val_dataset = train_val['test']

In [10]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [11]:
# TrainingArguments
training_args = TrainingArguments(
    output_dir="./model",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    report_to="none",
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.682700,0.304157,0.913366
2,0.503900,0.259194,0.928218
3,0.425700,0.299847,0.917492
4,0.360200,0.319507,0.920792
5,0.287000,0.313079,0.925743


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1140, training_loss=0.47444498915421335, metrics={'train_runtime': 3263.8598, 'train_samples_per_second': 16.71, 'train_steps_per_second': 0.349, 'total_flos': 1.4890930163514e+16, 'train_loss': 0.47444498915421335, 'epoch': 5.0})

In [14]:
test_dataset = Dataset.from_pandas(df_test)
test_dataset = test_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.remove_columns(["premise", "hypothesis", "id"])

Map:   0%|          | 0/5195 [00:00<?, ? examples/s]

In [15]:
predictions = trainer.predict(test_dataset)

In [16]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=1)

# 4. Build submission file
submission = pd.DataFrame({
    "id": df_test["id"],  # use the original IDs from test.csv
    "prediction": preds
})

submission.to_csv("submission.csv", index=False)